## Points to improve
- ~smoothen slope using 5th and 95th percentiles~
- ~convert decibels to linear~
- ~test code in Kashmir (valley)~
- push low level code to utils and other relevant modules
- final checks on documentation

In [33]:
import sys
sys.path.append('../')

from ifmiap import flood_mapper
from ifmiap import utils
import geopandas as gpd

In [34]:
# smal piece of code to get zone-wise list of IDs
gdf = gpd.read_file(r'../resources/india_utm_fishnet.gpkg')
zone_id_group = gdf[['zone', 'ID']].groupby('zone')['ID'].apply(list)

zone_id_dict = dict()

for idx in zone_id_group.index:
    zone_id_dict[idx] = zone_id_group[idx]

zone_id_dict['45R']

[338,
 339,
 340,
 341,
 342,
 343,
 344,
 345,
 346,
 347,
 348,
 349,
 350,
 351,
 352,
 353,
 354,
 355,
 356,
 357,
 358,
 359,
 360,
 361,
 362,
 363,
 364,
 365]

In [2]:
# create the flood mapper class
bihar_flood_mapper = flood_mapper(
    grid_shapefile = r'../resources/india_utm_fishnet.gpkg',
    grid_id_list = zone_id_dict['45R']#[378, 384, 390],
    dry_date_col = 'dry_month',
    id_col = 'ID',
    dry_years=[2019, 2019],
    slope_dir = r'../resources/slope/',
    wet_duration = ['2019/07', '2019/07']
)


Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 [378, 384, 390]


In [3]:
%%time
bihar_flood_mapper.get_dry_dates()

if len(bihar_flood_mapper.aoi_ids_to_process) > 0:
    bihar_flood_mapper.generate_dry_date_ranges()
    bihar_flood_mapper.get_s1_items(dry_wet='dry')
    bihar_flood_mapper.read_scenes(dry_wet='dry', overview_level=3)
    bihar_flood_mapper.generate_mean_std_by_aoi()
else:
    bihar_flood_mapper.load_mean_std_by_aoi()
    
bihar_flood_mapper.prepare_slope(dem_overview=1, buffer=500)
bihar_flood_mapper.prepare_wet_scenes(overview_level=3)
bihar_flood_mapper.generate_number_of_scenes(export_raster=True)
bihar_flood_mapper.map_floods(vv_thd=3, vh_thd=3, rel_slope_thd=20,
                              export_raster=False, export_vector=True, export_maps=False)
bihar_flood_mapper.merge_floods_by_date(export_raster=True)

Previously processed ../output/mean_std/2019_2019_aoi_378_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2019_2019_aoi_384_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2019_2019_aoi_390_vv_vh_mean_std.nc read successfully!
Slope for tile ID 378 found, will not be downloaded.
Slope for tile ID 384 found, will not be downloaded.
Slope for tile ID 390 found, will not be downloaded.


/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWa

/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/opt/homebrew/lib/python3.9/site-packages/rioxarray/raster_writer.py:130: UserWa

Flood cells not found in 384_S1B_IW_GRDH_1SDV_20190725T120450_20190725T120510_017290_020841_rtc.
CPU times: user 39.6 s, sys: 2.71 s, total: 42.3 s
Wall time: 5min 30s


In [4]:
#bihar_flood_mapper.flush_output(remove_slope=False)